# LeetCode 3637 — Trionic Array I bằng Python

Link đề: https://leetcode.com/problems/trionic-array-i/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
class Solution:
    def isTrionic(self, nums: List[int]) -> bool:
        ...
```

> Ghi chú: tên bài, đề, ví dụ và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng "trionic" là mảng có thể cắt bằng hai chỉ số `0 < p < q < n - 1` thành ba đoạn **liền kề, chung đầu mút**: `nums[0..p]` **tăng ngặt**, `nums[p..q]` **giảm ngặt**, `nums[q..n-1]` **tăng ngặt**.
2. Phân biệt **tăng ngặt** (`<`) với **không giảm** (`<=`), và hiểu hai đoạn kề nhau **dùng chung** phần tử `nums[p]`, `nums[q]`.
3. Nắm kỹ thuật quét một lượt với **con trỏ trượt qua ba giai đoạn** (tăng -> giảm -> tăng) và kiểm tra con trỏ chạm cuối mảng.
4. Viết 4 cách: vét cạn mọi cặp `(p, q)`, tiền xử lý để kiểm tra `O(1)`, quét ba con trỏ `O(n)`, và nén dấu hiệu bằng `groupby`; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: dùng `<=` thay `<`, cho phép đoạn rỗng (`p = 0` hoặc `q = n - 1`), quên kiểm tra tới cuối mảng, và đếm "số lần đổi chiều = 2" một cách ngây thơ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` độ dài `n`. Mảng được gọi là **trionic** nếu tồn tại các chỉ số `0 < p < q < n - 1` sao cho:

- `nums[0...p]` **tăng ngặt**,
- `nums[p...q]` **giảm ngặt**,
- `nums[q...n-1]` **tăng ngặt**.

Trả về `true` nếu `nums` là trionic, ngược lại `false`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,3,5,4,2,6]` | `true` | chọn `p = 2`, `q = 4`: `[1,3,5]` tăng, `[5,4,2]` giảm, `[2,6]` tăng |
| 2 | `[2,1,3]` | `false` | không có cách chọn `p`, `q` thỏa mãn |

Constraints: `3 <= n <= 100`, `-1000 <= nums[i] <= 1000`.

In [1]:
nums = [1, 3, 5, 4, 2, 6]
p, q = 2, 4
seg1, seg2, seg3 = nums[0:p + 1], nums[p:q + 1], nums[q:]
print(seg1, seg2, seg3)
print("tăng ngặt :", all(a < b for a, b in zip(seg1, seg1[1:])))
print("giảm ngặt :", all(a > b for a, b in zip(seg2, seg2[1:])))
print("tăng ngặt :", all(a < b for a, b in zip(seg3, seg3[1:])))

[1, 3, 5] [5, 4, 2] [2, 6]
tăng ngặt : True
giảm ngặt : True
tăng ngặt : True


## 3. Kiến thức nền

### 3.1. Tăng ngặt, giảm ngặt và so sánh cặp kề

Dãy **tăng ngặt** nếu mọi cặp kề `a, b` thỏa `a < b` (bằng nhau **không** được). **Giảm ngặt** nếu mọi cặp kề thỏa `a > b`. Để kiểm tra một đoạn chỉ cần xét các cặp kề nhau: `nums[i] < nums[i+1]` với mọi `i` trong đoạn.

In [2]:
def strictly_increasing(a):
    return all(x < y for x, y in zip(a, a[1:]))

print(strictly_increasing([1, 3, 5]))       # True
print(strictly_increasing([1, 3, 3]))       # False vì 3 == 3
print(strictly_increasing([7]))             # True (đoạn 1 phần tử: không có cặp nào)

True
False
True


### 3.2. Cắt lát (`slicing`) và `zip(a, a[1:])`

`a[l:r+1]` lấy các phần tử từ `l` đến `r` **gồm cả `r`** (vì cận phải loại trừ). `zip(a, a[1:])` ghép từng phần tử với phần tử ngay sau nó, tạo ra mọi cặp kề `(a[i], a[i+1])`. Hai đoạn kề nhau của đề **dùng chung** phần tử tại `p` và `q`, nên khi cắt hãy dùng `nums[0:p+1]`, `nums[p:q+1]`, `nums[q:]`.

In [3]:
a = [1, 3, 5, 4, 2, 6]
print(list(zip(a, a[1:])))
print(a[0:3], a[2:5], a[4:])

[(1, 3), (3, 5), (5, 4), (4, 2), (2, 6)]
[1, 3, 5] [5, 4, 2] [2, 6]


### 3.3. Dấu của hiệu hai phần tử kề (`sign`)

Với mỗi cặp kề, `sign = (x < y) - (x > y)` cho `+1` khi tăng, `-1` khi giảm, `0` khi bằng nhau (Python ép `True/False` thành `1/0`). Dãy dấu cho ta "hình dạng" của mảng: `[1,3,5,4,2,6]` có dấu `[+,+,-,-,+]`.

In [4]:
a = [1, 3, 5, 4, 2, 6]
signs = [(x < y) - (x > y) for x, y in zip(a, a[1:])]
print(signs)
print([(2 < 3) - (2 > 3), (3 < 3) - (3 > 3), (4 < 3) - (4 > 3)])

[1, 1, -1, -1, 1]
[1, 0, -1]


### 3.4. `itertools.groupby` — gom các phần tử liên tiếp giống nhau

`groupby(seq)` nhóm các phần tử **liền kề** bằng nhau thành từng nhóm `(khóa, các_phần_tử)`. Nếu chỉ lấy khóa, ta nén `[+,+,-,-,+]` thành `[+,-,+]` — đúng "ba giai đoạn" của đề.

In [5]:
from itertools import groupby
signs = [1, 1, -1, -1, 1]
print([k for k, _ in groupby(signs)])
print([(k, len(list(g))) for k, g in groupby(signs)])

[1, -1, 1]
[(1, 2), (-1, 2), (1, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy hình dung mảng như **ngọn núi rồi thung lũng rồi lại lên dốc**: lên (đỉnh `p`), xuống (đáy `q`), lên.

```
nums = [1, 3, 5, 4, 2, 6]
dấu    :  +  +  -  -  +
          ^^^^^  ^^^^  ^
          tăng   giảm  tăng     -> p = 2 (đỉnh), q = 4 (đáy)
```

Tính tay `[2, 1, 3]`: dấu `[-, +]` -> bắt đầu bằng **giảm**, không có đoạn tăng đầu tiên (đoạn `[0..p]` cần `p >= 1`) -> `false`.

Ba nhận xét quan trọng:

1. Mỗi đoạn phải có **ít nhất một bước** (một cặp kề), vì `0 < p < q < n - 1`: đoạn đầu có `p >= 1` bước, đoạn giữa có `q - p >= 1` bước, đoạn cuối có `n - 1 - q >= 1` bước.
2. Không được có hai phần tử kề **bằng nhau** ở bất kỳ đâu (`0` trong dãy dấu).
3. Dãy dấu phải đúng dạng `+...+ -...- +...+` (mỗi cụm khác rỗng) — và chỉ thế.

## 5. Cách 1 — Brute Force / baseline

Thử **mọi cặp** `0 < p < q < n - 1`, với mỗi cặp kiểm tra ba đoạn bằng hàm trợ giúp. Đúng theo định nghĩa, không cần nghĩ gì thêm.

Nút thắt: có `O(n²)` cặp và mỗi lần kiểm tra tốn `O(n)`, tổng `O(n³)`. Với `n <= 100` là khoảng `10^6` phép so sánh — vẫn qua được, nhưng lãng phí vì ta kiểm tra đi kiểm tra lại các đoạn giống nhau.

In [6]:
class SolutionBrute:
    def _inc(self, a, l, r):            # nums[l..r] tăng ngặt
        return all(a[i] < a[i + 1] for i in range(l, r))

    def _dec(self, a, l, r):            # nums[l..r] giảm ngặt
        return all(a[i] > a[i + 1] for i in range(l, r))

    def isTrionic(self, nums):
        n = len(nums)
        for p in range(1, n - 1):
            for q in range(p + 1, n - 1):
                if self._inc(nums, 0, p) and self._dec(nums, p, q) and self._inc(nums, q, n - 1):
                    return True
        return False

print(SolutionBrute().isTrionic([1, 3, 5, 4, 2, 6]))
print(SolutionBrute().isTrionic([2, 1, 3]))

True
False


**Độ phức tạp:** Time `O(n³)` (`O(n²)` cặp × `O(n)` kiểm tra), Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá 1:** `p` **bị ép** duy nhất. Vì `nums[p] > nums[p+1]` (đoạn giữa giảm ngặt và có ít nhất một bước) trong khi `nums[0..p]` tăng ngặt, nên `p` phải là **cuối của cụm tăng đầu tiên** (cụm tăng cực đại tính từ đầu). Tương tự `q` là **cuối của cụm giảm** ngay sau đó. Không cần thử mọi cặp.

**Bước đột phá 2:** do đó chỉ cần **một con trỏ `i`** đi từ trái sang phải qua ba giai đoạn:

1. Trong khi `nums[i] < nums[i+1]`: `i += 1`. Dừng ở `p = i`. Nếu `p == 0` -> sai (không có bước tăng nào).
2. Trong khi `nums[i] > nums[i+1]`: `i += 1`. Dừng ở `q = i`. Nếu `q == p` -> sai (không có bước giảm nào).
3. Trong khi `nums[i] < nums[i+1]`: `i += 1`. Cuối cùng cần `i == n - 1` (đi hết mảng) **và** `i > q` (có ít nhất một bước tăng cuối).

Invariant: mỗi lần con trỏ dừng thì phần tử kế tiếp **không** thuộc giai đoạn hiện tại (hoặc đã hết mảng) — nên điểm dừng chính là `p`/`q` duy nhất khả dĩ.

In [7]:
def trionic_pointer(nums):
    n = len(nums)
    i = 0
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1
    p = i
    if p == 0:
        return False
    while i + 1 < n and nums[i] > nums[i + 1]:
        i += 1
    q = i
    if q == p:
        return False
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1
    return i == n - 1 and q < n - 1

print(trionic_pointer([1, 3, 5, 4, 2, 6]), trionic_pointer([2, 1, 3]), trionic_pointer([1, 2, 1, 2]))

True False True


## 7. Các cách tiếp theo

### Cách 2 — Tiền xử lý để kiểm tra mỗi cặp `(p, q)` trong `O(1)`

Vẫn thử mọi cặp nhưng đã chuẩn bị sẵn:

- `up[i] = True` nếu `nums[0..i]` tăng ngặt (mảng tiền tố).
- `suf[i] = True` nếu `nums[i..n-1]` tăng ngặt (mảng hậu tố).
- `bad[i]` = số vị trí `j < i` mà `nums[j] <= nums[j+1]` (đếm cặp **không giảm** trong tiền tố). Đoạn `nums[p..q]` giảm ngặt khi `bad[q] - bad[p] == 0`.

Sau đó mỗi cặp kiểm tra bằng `up[p] and suf[q] and bad[q] == bad[p]`. Time `O(n²)`.

In [8]:
class SolutionPrefix:
    def isTrionic(self, nums):
        n = len(nums)
        up = [True] * n                       # up[i]: nums[0..i] tăng ngặt
        for i in range(1, n):
            up[i] = up[i - 1] and nums[i - 1] < nums[i]
        suf = [True] * n                      # suf[i]: nums[i..n-1] tăng ngặt
        for i in range(n - 2, -1, -1):
            suf[i] = suf[i + 1] and nums[i] < nums[i + 1]
        bad = [0] * n                         # bad[i]: số j < i có nums[j] <= nums[j+1]
        for i in range(1, n):
            bad[i] = bad[i - 1] + (1 if nums[i - 1] <= nums[i] else 0)
        for p in range(1, n - 1):
            if not up[p]:
                continue
            for q in range(p + 1, n - 1):
                if suf[q] and bad[q] == bad[p]:
                    return True
        return False

print(SolutionPrefix().isTrionic([1, 3, 5, 4, 2, 6]), SolutionPrefix().isTrionic([2, 1, 3]))

True False


**Độ phức tạp:** Time `O(n²)` (hai vòng lặp, mỗi cặp `O(1)`), Space `O(n)` cho ba mảng phụ.

### Cách 3 — Quét ba con trỏ `O(n)` (cách nộp bài)

Chính là thuật toán ở mục 6: chỉ một biến `i`, đi qua ba giai đoạn rồi kiểm tra `i == n - 1`. Không cần mảng phụ.

In [9]:
class SolutionPointer:
    def isTrionic(self, nums):
        n = len(nums)
        i = 0
        while i + 1 < n and nums[i] < nums[i + 1]:
            i += 1
        if i == 0:
            return False
        p = i
        while i + 1 < n and nums[i] > nums[i + 1]:
            i += 1
        if i == p:
            return False
        q = i
        while i + 1 < n and nums[i] < nums[i + 1]:
            i += 1
        return i == n - 1 and q < n - 1

print(SolutionPointer().isTrionic([1, 3, 5, 4, 2, 6]), SolutionPointer().isTrionic([2, 1, 3]))

True False


**Độ phức tạp:** Time `O(n)` (con trỏ `i` chỉ đi **một chiều**, mỗi phần tử được xét `O(1)` lần), Space `O(1)`.

### Cách 4 — Nén dãy dấu bằng `groupby`

Tính dãy dấu của các cặp kề. Nếu có `0` (hai phần tử bằng nhau) -> `False`. Ngược lại nén các dấu liên tiếp giống nhau; mảng là trionic **khi và chỉ khi** kết quả nén chính xác là `[+1, -1, +1]`. Cách này trực quan: "ba giai đoạn" = "ba nhóm dấu".

In [10]:
from itertools import groupby

class SolutionGroupby:
    def isTrionic(self, nums):
        signs = [(a < b) - (a > b) for a, b in zip(nums, nums[1:])]
        if 0 in signs:
            return False
        return [k for k, _ in groupby(signs)] == [1, -1, 1]

print(SolutionGroupby().isTrionic([1, 3, 5, 4, 2, 6]), SolutionGroupby().isTrionic([2, 1, 3]))

True False


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho danh sách dấu (có thể dùng generator để còn `O(1)`).

## 8. TRACE từng bước

Trace `nums = [1, 3, 5, 4, 2, 6]` bằng thuật toán con trỏ (`n = 6`).

| Bước | giai đoạn | i trước | so sánh | i sau |
|---|---|---|---|---|
| 1 | tăng | 0 | `1 < 3` | 1 |
| 2 | tăng | 1 | `3 < 5` | 2 |
| 3 | tăng | 2 | `5 < 4`? không -> dừng, `p = 2` | 2 |
| 4 | giảm | 2 | `5 > 4` | 3 |
| 5 | giảm | 3 | `4 > 2` | 4 |
| 6 | giảm | 4 | `2 > 6`? không -> dừng, `q = 4` | 4 |
| 7 | tăng | 4 | `2 < 6` | 5 |
| 8 | tăng | 5 | `i + 1 = n` -> dừng, `i = 5 = n - 1` | 5 |

`p = 2 > 0`, `q = 4 > p`, `i == n - 1` và `q < n - 1` nên **true**.

In [11]:
def trace_trionic(nums):
    n = len(nums)
    print("nums =", nums)
    i = 0
    for name, ok in (("TĂNG", lambda a, b: a < b), ("GIẢM", lambda a, b: a > b), ("TĂNG", lambda a, b: a < b)):
        start = i
        while i + 1 < n and ok(nums[i], nums[i + 1]):
            print(f"  [{name}] i={i}: {nums[i]} -> {nums[i + 1]}")
            i += 1
        print(f"  => giai đoạn {name} chạy từ {start} đến {i}")
        if i == start:
            print("  => giai đoạn rỗng -> False")
            return False
    print(f"  i={i}, n-1={n - 1}, ->", i == n - 1)
    return i == n - 1

trace_trionic([1, 3, 5, 4, 2, 6])
trace_trionic([2, 1, 3])
trace_trionic([1, 3, 2, 4, 3])

nums = [1, 3, 5, 4, 2, 6]
  [TĂNG] i=0: 1 -> 3
  [TĂNG] i=1: 3 -> 5
  => giai đoạn TĂNG chạy từ 0 đến 2
  [GIẢM] i=2: 5 -> 4
  [GIẢM] i=3: 4 -> 2
  => giai đoạn GIẢM chạy từ 2 đến 4
  [TĂNG] i=4: 2 -> 6
  => giai đoạn TĂNG chạy từ 4 đến 5
  i=5, n-1=5, -> True
nums = [2, 1, 3]
  => giai đoạn TĂNG chạy từ 0 đến 0
  => giai đoạn rỗng -> False
nums = [1, 3, 2, 4, 3]
  [TĂNG] i=0: 1 -> 3
  => giai đoạn TĂNG chạy từ 0 đến 1
  [GIẢM] i=1: 3 -> 2
  => giai đoạn GIẢM chạy từ 1 đến 2
  [TĂNG] i=2: 2 -> 4
  => giai đoạn TĂNG chạy từ 2 đến 3
  i=3, n-1=4, -> False


False

## 9. Code nộp LeetCode

In [12]:
from typing import List, Optional

class Solution:
    def isTrionic(self, nums: List[int]) -> bool:
        n = len(nums)
        i = 0
        # giai đoạn 1: tăng ngặt
        while i + 1 < n and nums[i] < nums[i + 1]:
            i += 1
        if i == 0:
            return False
        p = i
        # giai đoạn 2: giảm ngặt
        while i + 1 < n and nums[i] > nums[i + 1]:
            i += 1
        if i == p:
            return False
        q = i
        # giai đoạn 3: tăng ngặt, phải tới hết mảng và có ít nhất một bước
        while i + 1 < n and nums[i] < nums[i + 1]:
            i += 1
        return i == n - 1 and q < n - 1

## 10. Vì sao thuật toán đúng

- **Điều kiện cần:** nếu mảng trionic với `(p, q)` thì `nums[p] > nums[p+1]` (đoạn giữa có `q > p` nên có ít nhất một bước giảm). Cùng với `nums[0..p]` tăng ngặt, điều này nghĩa `p` đúng là điểm kết thúc của cụm tăng cực đại từ đầu — chính là điểm con trỏ dừng ở giai đoạn 1. Tương tự `q` là điểm dừng ở giai đoạn 2, vì `nums[q] < nums[q+1]` (đoạn cuối có `q < n - 1`).
- **Điều kiện đủ:** nếu con trỏ dừng ở `p > 0`, rồi `q > p`, rồi đi hết mảng đến `n - 1` với `q < n - 1`, thì ba đoạn `[0..p]`, `[p..q]`, `[q..n-1]` lần lượt tăng, giảm, tăng ngặt theo cách xây dựng vòng `while`.
- **Ba kiểm tra phụ** loại ba lỗi: `p == 0` (không có đoạn tăng đầu), `q == p` (không có đoạn giảm), `q == n - 1` (không có đoạn tăng cuối). Kiểm tra `i == n - 1` loại trường hợp còn "đuôi lạ" sau giai đoạn 3 (ví dụ `[1,3,2,4,3]`).
- **Bằng nhau:** vì dùng `<` và `>` ngặt, hai phần tử bằng nhau làm con trỏ dừng sớm ở giai đoạn nào đó, và đến giai đoạn cuối `i != n - 1` hoặc có đoạn rỗng -> `False`.

## 11. Phân tích Time / Space Complexity

`n` = `len(nums)` (`<= 100`).

| Cách | Time | Space |
|---|---|---|
| Brute force mọi `(p, q)` | `O(n³)` | `O(1)` |
| Tiền xử lý + mọi cặp | `O(n²)` | `O(n)` |
| Quét ba con trỏ | `O(n)` | `O(1)` |
| `groupby` dấu | `O(n)` | `O(n)` |

Với giới hạn `n <= 100` mọi cách đều qua; nhưng `O(n)` / `O(1)` là cách chuẩn, và cũng sẽ cần cho bài tiếp theo (Trionic Array II) khi mảng lớn tới `10^5`.

## 12. Bộ test

Ví dụ của đề, các ca biên và phản ví dụ, sau đó đối chiếu **tất cả cách** trên hàng nghìn mảng ngẫu nhiên.

In [13]:
impls = [SolutionBrute(), SolutionPrefix(), SolutionPointer(), SolutionGroupby(), Solution()]

cases = [
    ([1, 3, 5, 4, 2, 6], True),
    ([2, 1, 3], False),
    ([1, 2, 1], False),                 # n = 3 không bao giờ trionic
    ([1, 3, 2, 4], True),               # p=1, q=2
    ([1, 3, 3, 2, 4], False),           # bằng nhau -> không ngặt
    ([3, 2, 1, 2], False),              # thiếu đoạn tăng đầu
    ([1, 2, 3, 2], False),              # thiếu đoạn tăng cuối
    ([1, 3, 2, 4, 3], False),           # còn đuôi giảm sau đoạn tăng cuối
    ([1, 2, 3, 4, 5], False),           # chỉ tăng
    ([5, 4, 3, 2, 1], False),           # chỉ giảm
    ([3, 2, 3, 2], False),              # xuống-lên-xuống: 2 lần đổi chiều nhưng sai dạng
    ([-1000, 1000, -1000, 1000], True), # biên giá trị
]
for nums, exp in cases:
    for impl in impls:
        got = impl.isTrionic(list(nums))
        assert got == exp, (type(impl).__name__, nums, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 12 ca


In [14]:
import random

def brute_ref(a):
    n = len(a)
    for p in range(1, n - 1):
        for q in range(p + 1, n - 1):
            if all(a[i] < a[i + 1] for i in range(0, p)) and all(a[i] > a[i + 1] for i in range(p, q)) \
               and all(a[i] < a[i + 1] for i in range(q, n - 1)):
                return True
    return False

random.seed(3637)
true_cnt = 0
for _ in range(3000):
    n = random.randint(3, 10)
    a = [random.randint(-3, 3) for _ in range(n)]          # giá trị nhỏ -> nhiều ca bằng nhau
    exp = brute_ref(a)
    true_cnt += exp
    for impl in impls:
        assert impl.isTrionic(a[:]) == exp, (a, type(impl).__name__)
print("3000 mảng ngẫu nhiên nhỏ OK, số mảng trionic:", true_cnt)
assert true_cnt > 0

3000 mảng ngẫu nhiên nhỏ OK, số mảng trionic: 153


In [15]:
# Sinh mảng trionic hợp lệ theo đúng constraints (n <= 100, |x| <= 1000) và mảng đột biến
def make_trionic(n):
    while True:
        a, b = sorted(random.sample(range(1, n - 1), 2))   # p = a, q = b, 0 < p < q < n-1
        vals = [0] * n
        vals[0] = random.randint(-1000, -600)
        for i in range(1, a + 1):
            vals[i] = vals[i - 1] + random.randint(1, 5)
        for i in range(a + 1, b + 1):
            vals[i] = vals[i - 1] - random.randint(1, 5)
        for i in range(b + 1, n):
            vals[i] = vals[i - 1] + random.randint(1, 5)
        if all(-1000 <= v <= 1000 for v in vals):
            return vals

random.seed(2024)
for _ in range(300):
    n = random.randint(4, 100)
    a = make_trionic(n)
    assert Solution().isTrionic(a) is True
    assert all(impl.isTrionic(a[:]) is True for impl in impls[1:])
    b = a[:]
    k = random.randrange(0, n - 1)
    b[k + 1] = b[k]                                        # tạo hai phần tử bằng nhau -> không ngặt
    assert Solution().isTrionic(b) is False and brute_ref(b) is False
print("300 mảng trionic dài + 300 mảng bị phá OK")

300 mảng trionic dài + 300 mảng bị phá OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `<=` / `>=` thay cho `<` / `>`

Đề yêu cầu **ngặt**; dùng `<=` cho phép các phần tử bằng nhau.

In [16]:
def wrong_nonstrict(nums):
    n = len(nums)
    i = 0
    while i + 1 < n and nums[i] <= nums[i + 1]:        # SAI: không ngặt
        i += 1
    if i == 0:
        return False
    p = i
    while i + 1 < n and nums[i] >= nums[i + 1]:
        i += 1
    if i == p:
        return False
    q = i
    while i + 1 < n and nums[i] <= nums[i + 1]:
        i += 1
    return i == n - 1 and q < n - 1

a = [1, 3, 3, 2, 4]
print(wrong_nonstrict(a), "!=", Solution().isTrionic(a))
assert wrong_nonstrict(a) != Solution().isTrionic(a)       # True != False

True != False


### Lỗi 2: cho phép đoạn rỗng (`p = 0` hoặc `q = n - 1`)

Điều kiện `0 < p < q < n - 1` bắt buộc **mỗi đoạn có ít nhất một bước**. Bỏ kiểm tra `i == 0` làm `[3,2,1,2]` bị nhận nhầm.

In [17]:
def wrong_allow_empty(nums):
    n = len(nums)
    i = 0
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1                                          # SAI: không kiểm tra i == 0
    while i + 1 < n and nums[i] > nums[i + 1]:
        i += 1
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1
    return i == n - 1

a = [3, 2, 1, 2]
print(wrong_allow_empty(a), "!=", Solution().isTrionic(a))
assert wrong_allow_empty(a) != Solution().isTrionic(a)     # True != False

True != False


### Lỗi 3: quên kiểm tra đã đi hết mảng

Sau giai đoạn tăng cuối, nếu còn phần tử thừa (ví dụ lại giảm) thì mảng **không** trionic. Trả `True` ngay khi qua ba giai đoạn là sai.

In [18]:
def wrong_no_end_check(nums):
    n = len(nums)
    i = 0
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1
    if i == 0:
        return False
    p = i
    while i + 1 < n and nums[i] > nums[i + 1]:
        i += 1
    if i == p:
        return False
    q = i
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1
    return i > q                                        # SAI: thiếu i == n - 1

a = [1, 3, 2, 4, 3]
print(wrong_no_end_check(a), "!=", Solution().isTrionic(a))
assert wrong_no_end_check(a) != Solution().isTrionic(a)    # True != False

True != False


### Lỗi 4: "đúng 2 lần đổi chiều" là trionic

Hai lần đổi chiều có thể là `+ - +` (đúng) **hoặc** `- + -` (sai). Phải kiểm tra **dạng** `+,-,+`, không chỉ đếm số lần đổi.

In [19]:
def wrong_count_turns(nums):
    signs = [(a < b) - (a > b) for a, b in zip(nums, nums[1:])]
    if 0 in signs:
        return False
    turns = sum(1 for x, y in zip(signs, signs[1:]) if x != y)
    return turns == 2                                  # SAI: thiếu kiểm tra dạng + - +

a = [3, 2, 3, 2]
print(wrong_count_turns(a), "!=", Solution().isTrionic(a))
assert wrong_count_turns(a) != Solution().isTrionic(a)     # True != False

True != False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tăng rồi giảm rồi tăng" / "núi" / "thung lũng" -> **con trỏ trượt qua các giai đoạn**, mỗi giai đoạn là một vòng `while` (cùng mẫu với Valid Mountain Array, LeetCode 941).
- Gặp điều kiện "ngặt" nhớ `<` / `>`; gặp "không giảm" mới là `<=`.
- Đoạn con **dùng chung đầu mút** thì khi cắt lát nhớ `+1`.
- Luôn tự hỏi: mỗi đoạn có được **rỗng** không? Ở đây không, vì `0 < p < q < n - 1`.
- Sau các vòng `while` phải xác nhận con trỏ **đã tới cuối** (`i == n - 1`); nếu không, phần đuôi chưa kiểm tra.
- Khi bí, vẽ dãy dấu `+/-/0`: hình dạng mảng hiện ra ngay.
- `n = 3` luôn `False` (không có `p < q` trong `1 <= p < q <= 1`).

## 15. Biến thể của bài

1. **Trionic Array II** (LeetCode 3640, Hard): chọn mảng con trionic có tổng lớn nhất — cần quy hoạch động hoặc quét các "đoạn" thay vì chỉ trả `true/false`.
2. **Valid Mountain Array** (941): tăng rồi giảm, mỗi đoạn có ít nhất một bước.
3. **Monotonic Array** (896): tăng hoặc giảm không ngặt toàn bộ.
4. Đếm số **cặp `(p, q)`** hợp lệ khi bỏ điều kiện "ngặt" (dùng hai cụm bằng nhau).
5. Tổng quát hóa: mảng có đúng `k` đoạn đổi chiều theo dạng `+ - + - ...` (nén dấu rồi so sánh với mẫu).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Brute force | thử mọi `(p, q)`, kiểm tra ba đoạn | `O(n³)` | `O(1)` | Để hiểu đề, viết test đối chiếu |
| Tiền xử lý | mảng `up`, `suf`, `bad` rồi thử mọi cặp | `O(n²)` | `O(n)` | Khi muốn kiểm tra đoạn bất kỳ trong `O(1)` |
| Quét ba con trỏ | một biến `i`, ba vòng `while` | `O(n)` | `O(1)` | Cách nộp chuẩn |
| `groupby` dấu | nén dấu thành `[+,-,+]` | `O(n)` | `O(n)` | Dễ đọc, hợp khi pattern phức tạp hơn |

Chọn quét ba con trỏ để nộp; dùng brute force làm "chuẩn" khi viết test.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().isTrionic([1, 4, 2, 3])` là gì?

**Bài 2 (tính tay):** viết dãy dấu của `[5, 6, 7, 3, 2, 8, 9]` và cho biết có trionic không, `p`, `q` bằng bao nhiêu.

**Bài 3 (điền TODO):** hoàn thành `sign_list(nums)` trả danh sách dấu `+1/-1/0` của các cặp kề.

**Bài 4 (tự viết lại):** viết `is_mountain(nums)` (LeetCode 941): tăng ngặt rồi giảm ngặt, mỗi đoạn có ít nhất một bước, `len >= 3`.

**Bài 5 (mini challenge):** viết `find_pq(nums)` trả về cặp `(p, q)` nếu `nums` là trionic, ngược lại `None`.

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().isTrionic([1, 4, 2, 3]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def sign_list_todo(nums):
    # TODO: với mỗi cặp kề (a, b): +1 nếu a < b, -1 nếu a > b, 0 nếu bằng
    return []

True


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: dấu `[+, -, +]`. Bài 2: tính hiệu từng cặp kề từ trái sang phải.

**Mức 2 (vừa):** Bài 3: `(a < b) - (a > b)`. Bài 4: giống mục 9 nhưng chỉ có **hai** giai đoạn, và cuối cùng `i == n - 1` với `p > 0` và `i > p`. Bài 5: sao chép quét ba con trỏ và trả `(p, q)` thay vì `bool`.

**Mức 3 (gần đáp án):** Bài 2: dấu `[+, +, -, -, +, +]` -> `p = 2`, `q = 4`, `true`. Bài 5: sau khi chạy xong ba vòng và kiểm tra hợp lệ, `return (p, q)`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1: dấu [+, -, +] -> p=1, q=2
assert Solution().isTrionic([1, 4, 2, 3]) is True

# Bài 2: [5,6,7,3,2,8,9] dấu [+,+,-,-,+,+] -> p=2, q=4
nums2 = [5, 6, 7, 3, 2, 8, 9]
assert Solution().isTrionic(nums2) is True

# Bài 3
def sign_list(nums):
    return [(a < b) - (a > b) for a, b in zip(nums, nums[1:])]
assert sign_list(nums2) == [1, 1, -1, -1, 1, 1]
assert sign_list([2, 2, 1]) == [0, -1]

# Bài 4
def is_mountain(nums):
    n = len(nums)
    i = 0
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1
    if i == 0 or i == n - 1:
        return False
    while i + 1 < n and nums[i] > nums[i + 1]:
        i += 1
    return i == n - 1
assert is_mountain([0, 3, 2, 1]) is True
assert is_mountain([3, 5, 5]) is False
assert is_mountain([2, 1]) is False

# Bài 5
def find_pq(nums):
    n = len(nums)
    i = 0
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1
    if i == 0:
        return None
    p = i
    while i + 1 < n and nums[i] > nums[i + 1]:
        i += 1
    if i == p:
        return None
    q = i
    while i + 1 < n and nums[i] < nums[i + 1]:
        i += 1
    return (p, q) if i == n - 1 and q < n - 1 else None
assert find_pq([1, 3, 5, 4, 2, 6]) == (2, 4)
assert find_pq(nums2) == (2, 4)
assert find_pq([2, 1, 3]) is None
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "con trỏ trượt qua các giai đoạn tăng / giảm":**

```python
n, i = len(nums), 0
while i + 1 < n and nums[i] < nums[i + 1]: i += 1     # tăng
if i == 0: return False
p = i
while i + 1 < n and nums[i] > nums[i + 1]: i += 1     # giảm
if i == p: return False
q = i
while i + 1 < n and nums[i] < nums[i + 1]: i += 1     # tăng
return i == n - 1 and q < n - 1
```

**Checklist trước khi submit:**

- [ ] Dùng so sánh **ngặt** `<`, `>` (không `<=`, `>=`)?
- [ ] Mỗi đoạn có **ít nhất một bước** (`p > 0`, `q > p`, `q < n - 1`)?
- [ ] Sau giai đoạn cuối có `i == n - 1` (không còn đuôi thừa)?
- [ ] Đã thử `[2,1,3]`, `[1,3,3,2,4]`, `[3,2,1,2]`, `[1,3,2,4,3]`, và `n = 3`?
- [ ] Biên `i + 1 < n` trong mọi vòng `while` để không tràn chỉ số?
- [ ] Chữ ký đúng `isTrionic(self, nums: List[int]) -> bool`?